# Entity ***Swaps***
+ Layer **gold**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
sv_swaps_df = spark.read.table("uniswap.silver.swaps")

In [0]:
gld_swaps_df = spark.sql(f"""
    WITH cte_silver_swaps AS (
        SELECT
            pk_swap_id,
            pool_id AS fk_pool_contract,
            token0_id AS fk_token0_contract,
            token1_id AS fk_token1_contract,
            transaction_id,
            sender_address,
            recipient_address,
            amount_USD,
            amount_token0,
            amount_token1,
            after_swap_tick_idx,
            block_timestamp
        FROM {{df}}
    )
    SELECT
        *,
        CASE
            WHEN amount_USD >= 100000 THEN "whale"
            WHEN amount_USD >= 10000 THEN "large"
            WHEN amount_USD >= 1000 THEN "medium"
            WHEN amount_USD IS NULL THEN "unknown"
            ELSE "small"
        END AS swap_tier,
        CURRENT_TIMESTAMP() AS _load_timestamp_gold
    FROM cte_silver_swaps
""", df = sv_swaps_df)

### Save & exit

In [0]:
load_result = load_entity(gld_swaps_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)